# 세그먼트 분리 실험 — 로컬 VS Code (STEP0 → A/B)
**환경:** 로컬(M1, GPU 없음). 상대경로. NN 재학습 없음 — 트리(lgb)/OOF 기반.
**질문:** 운영 블렌드가 *이식배아=0 deterministic 구간*을 이미 바닥에 깔았나? → 분기.
- **흡수** → STEP B: 이식>0 영역 천장 재측정(우승팀 동력 축).
- **미흡수** → STEP A: 세그먼트 모델(이식>0만 학습 + 이식=0 상수).

**안전장치:** ① OOF 블렌드 재구성 시 **y(라벨)·id 컬럼 제외**(제안서 `.mean()` 오염 회피) + AUC<0.999 멤버만. ② y 정렬 assert. ③ 분리는 *알려진 컬럼*(이식배아 수)만, 타깃 미사용. ④ seg_zero 상수는 train 통계만.
**입력(로컬, `INPUT_DIR`):** `train.csv` + 운영 멤버 OOF — `oof_predictions.csv`(lgb/cat/xgb/lin base+y), `oof_v2_trees.csv`(v2_lgb/cat/xgb), `oof_lin_ratio.csv`, `oof_nn.csv`. 운영 best(0.74209) 재구성용.

## 0. 설정 (로컬 경로)

In [1]:
import os, glob, re, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import lightgbm as lgb
warnings.filterwarnings("ignore")
SEEDS = (42, 7, 2024)
# 데이터 폴더 자동 탐색: cwd가 노트북 폴더(notebooks/lee)든 프로젝트 루트든 무관하게 train.csv 있는 data/ 찾음
def _find_data_dir():
    for d in ["./data","../data","../../data","../../../data","data"]:
        if os.path.exists(os.path.join(d,"train.csv")): return d
    cur=os.path.abspath(os.getcwd())
    for _ in range(6):
        p=os.path.join(cur,"data","train.csv")
        if os.path.exists(p): return os.path.join(cur,"data")
        cur=os.path.dirname(cur)
    return "./data"
INPUT_DIR = _find_data_dir()   # ← 자동 탐색. 필요시 수동 지정 가능.
_SEARCH = [INPUT_DIR, ".", "./data", "../data", "../../data"]
def P(name):
    for d in _SEARCH:
        p=os.path.join(d,name)
        if os.path.exists(p): return p
    raise FileNotFoundError(f"{name} 없음 — INPUT_DIR={INPUT_DIR} 확인")
def P_safe(name):
    for d in _SEARCH:
        p=os.path.join(d,name)
        if os.path.exists(p): return p
    return os.path.join(INPUT_DIR,name)
train = pd.read_csv(P("train.csv"))
TARGET="임신 성공 여부"; ID_COL="ID"
y = train[TARGET].astype(int).values; N=len(train); base_rate=y.mean()
tx = pd.to_numeric(train["이식된 배아 수"], errors="coerce").fillna(0).values
m0 = (tx==0); mpos = ~m0
def runr(x): return rankdata(x)/len(x)
print(f"INPUT_DIR={INPUT_DIR}")
print(f"train {train.shape} | base_rate={base_rate:.4f}")
print(f"이식=0: n={m0.sum()} ({100*m0.mean():.1f}%) 성공률={y[m0].mean():.4f} | 이식>0: n={mpos.sum()} 성공률={y[mpos].mean():.4f}")

INPUT_DIR=../../data
train (256351, 69) | base_rate=0.2583
이식=0: n=42835 (16.7%) 성공률=0.0196 | 이식>0: n=213516 성공률=0.3062


## 1. STEP 0 — 진단: 이식배아=0이 운영 블렌드에 이미 흡수됐나
판정 핵심: **이식>0 부분집합 AUC < 전체 AUC** 면 deterministic 구간이 전체 AUC를 띄우는 것 = 진짜 천장은 이식>0 위.

In [2]:
# ── 운영 best OOF 재구성 (단일 파일 없음 → 멤버 OOF 가중 합성). 세 기준 비교.
def L(name):  # 로컬 로드 + 정렬 검증 (y 있으면 y로, 없고 id 있으면 id로 — v2는 y가 없어 id로 검증)
    d=pd.read_csv(P(name))
    if "y" in d.columns: assert np.array_equal(d["y"].astype(int).values,y), f"{name} y 불일치 — 정렬 깨짐!"
    elif "id" in d.columns:
        # v2 등은 id가 정수 위치인덱스(0..N-1), train ID는 'TRAIN_000000' 문자열 → 숫자만 뽑아 위치 대조
        tkey=pd.Series(train[ID_COL].astype(str)).str.extract(r"(\d+)")[0].astype("int64").values
        assert np.array_equal(np.asarray(d["id"]).astype("int64"),tkey), f"{name} id 정렬 깨짐 — train 순서와 불일치!"
    return d
op=L("oof_predictions.csv")                 # lgb_base 등: oof_lgb/oof_cat/oof_xgb/oof_lr
v2_=L("oof_v2_trees.csv") if os.path.exists(P_safe("oof_v2_trees.csv")) else None
lr_=L("oof_lin_ratio.csv") if os.path.exists(P_safe("oof_lin_ratio.csv")) else None
nn_=L("oof_nn.csv") if os.path.exists(P_safe("oof_nn.csv")) else None
# base 멤버 라벨 오로드 가드
for cc in ["oof_lgb","oof_cat","oof_xgb","oof_lr"]:
    if cc in op: assert roc_auc_score(y,op[cc])<0.999, f"{cc} 라벨 오로드 의심"
def view_col(view, model):
    if view=="base": return op["oof_lr"].values if model=="lin" else op[f"oof_{model}"].values
    if view=="v2":   return v2_[f"v2_{model}"].values
    if view=="ratio":return lr_["oof_lin_ratio"].values
    raise KeyError(view)
def recon(combo, w):
    parts=[w[m]*runr(view_col(combo[m],m)) for m in ["lgb","xgb","cat"]]
    parts.append(w["lin"]*runr(view_col(combo["lin"],"lin")))
    parts.append(w["nn"]*runr(nn_["oof_nn"].values))   # nn=base (운영 best 둘 다 nn=base)
    return np.sum(parts,axis=0)
# 정확한 조합·가중치 (그리드 로그 c25/c27 원문)
SUBMIT =({"lgb":"v2","xgb":"v2","cat":"v2","lin":"ratio"},   {"lgb":0.27,"xgb":0.17,"cat":0.30,"lin":0.03,"nn":0.23})  # 0.74209 LB / 로컬 OOF 0.74056
OOFBEST=({"lgb":"v2","xgb":"base","cat":"v2","lin":"ratio"}, {"lgb":0.27,"xgb":0.20,"cat":0.28,"lin":0.03,"nn":0.22})  # 0.74058 (제출X: xgb_base test 부재)
def base_blend():  # all-base 동일가중 (참조)
    return np.mean([runr(op[c].values) for c in ["oof_lgb","oof_cat","oof_xgb"]],axis=0)

CRITERIA={}
if v2_ is not None and lr_ is not None and nn_ is not None:
    CRITERIA["제출best(0.74209)"]=recon(*SUBMIT)
    CRITERIA["OOFbest(0.74058)"]=recon(*OOFBEST)
CRITERIA["base 3트리"]=base_blend()
# 재구성 검증: 멤버 정렬/가중치가 틀어지면 AUC가 붕괴 → 조기 탐지 (LB≠OOF지만 로컬 OOF는 ~0.7406 부근이어야 정상)
if "제출best(0.74209)" in CRITERIA:
    _ar=roc_auc_score(y,CRITERIA["제출best(0.74209)"])
    assert 0.730<_ar<0.745, f"재구성 제출best OOF AUC {_ar:.5f} 비정상 — 멤버 정렬(v2 id?)/가중치 확인"

print("=== STEP0: 운영 기준별 이식=0 흡수 진단 ===")
auc_pos_by={}
for name,p_op in CRITERIA.items():
    pct=rankdata(p_op)/len(p_op)
    a_all=roc_auc_score(y,p_op); a_pos=roc_auc_score(y[mpos],p_op[mpos]); auc_pos_by[name]=a_pos
    med=np.median(pct[m0]); low10=(pct[m0]<0.10).mean()
    absorbed=med<0.15
    print(f"\n[{name}]")
    print(f"  전체 AUC {a_all:.5f} | 이식>0 부분집합 {a_pos:.5f} (차 {a_all-a_pos:+.5f})")
    print(f"  이식=0 OOF 평균 {p_op[m0].mean():.4f} | 백분위중앙값 {med:.3f} | 하위10%비율 {low10:.3f} → {'흡수' if absorbed else '미흡수'}")
# 운영 기준(제출best 우선) 채택
KEY="제출best(0.74209)" if "제출best(0.74209)" in CRITERIA else "base 3트리"
p_op=CRITERIA[KEY]; pct=rankdata(p_op)/len(p_op)
auc_all=roc_auc_score(y,p_op); auc_pos=auc_pos_by[KEY]
ABSORBED=np.median(pct[m0])<0.15
print(f"\n→ 운영 기준 = {KEY} | 흡수={ABSORBED} | 권고:", "STEP B(천장 재측정)" if ABSORBED else "STEP A(세그먼트 모델)")
print("  (기준 셋 다 흡수 결론 같으면 base/v2 차이는 진단에 무영향 = 견고)")

=== STEP0: 운영 기준별 이식=0 흡수 진단 ===

[제출best(0.74209)]
  전체 AUC 0.74056 | 이식>0 부분집합 0.67456 (차 +0.06599)
  이식=0 OOF 평균 0.0993 | 백분위중앙값 0.084 | 하위10%비율 0.598 → 흡수



[OOFbest(0.74058)]
  전체 AUC 0.74058 | 이식>0 부분집합 0.67459 (차 +0.06599)
  이식=0 OOF 평균 0.0993 | 백분위중앙값 0.084 | 하위10%비율 0.598 → 흡수

[base 3트리]
  전체 AUC 0.74010 | 이식>0 부분집합 0.67396 (차 +0.06615)
  이식=0 OOF 평균 0.0982 | 백분위중앙값 0.084 | 하위10%비율 0.598 → 흡수

→ 운영 기준 = 제출best(0.74209) | 흡수=True | 권고: STEP B(천장 재측정)
  (기준 셋 다 흡수 결론 같으면 base/v2 차이는 진단에 무영향 = 견고)


## 2. 정본 tf_tree (train-only fit, 누수 안전) + 세그먼트 학습기

In [3]:
COL_PROC="특정 시술 유형"; COL_RSN="배아 생성 주요 이유"
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
OCC=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수","총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
CMAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
_tp=lambda s: [] if pd.isna(s) else [t.strip() for t in re.split(r"[/:]",str(s)) if t.strip()]
def fit_tree(tr):
    st={}; ig={TARGET,ID_COL}
    st["dead"]=[c for c in tr.columns if c not in ig and tr[c].nunique(dropna=True)<=1]
    st["sparse"]=[c for c in tr.columns if c not in ig and c not in st["dead"] and tr[c].isna().mean()>0.98]
    st["lc"]={c:pd.Index(tr[c].astype("category").cat.categories) for c in NOMINAL_COLS if c in tr}
    st["pv"]=sorted({t for L in tr[COL_PROC].apply(_tp) for t in L}) if COL_PROC in tr else []
    return st
def tf_tree(df,st):
    df=df.copy()
    if TARGET in df: df=df.drop(columns=[TARGET])
    df["is_DI"]=(df["시술 유형"]=="DI").astype(int)
    df=df.drop(columns=[c for c in st["dead"] if c in df.columns])
    for c in st["sparse"]:
        if c in df: df[f"{c}_있음"]=df[c].notna().astype(int); df=df.drop(columns=[c])
    for c in OCC:
        if c in df: df[c]=df[c].astype(object).map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].astype(object).map(m)
    cats=[]
    for c,cc in st["lc"].items():
        if c in df: df[c]=pd.Categorical(df[c],categories=cc).codes.astype("int32"); cats.append(c)
    if COL_PROC in df:
        ts=df[COL_PROC].apply(_tp)
        for v in st["pv"]: df[f"proc_{v}"]=ts.apply(lambda L,v=v:int(v in L))
    df=df.drop(columns=[c for c in [COL_PROC,COL_RSN,ID_COL] if c in df.columns])
    obj=[c for c in df.columns if df[c].dtype==object]
    if obj: df=df.drop(columns=obj)
    for c in cats: df[c]=df[c].fillna(-1).astype("int32")
    return df,[c for c in cats if c in df.columns]
st=fit_tree(train); Xb,CATF=tf_tree(train,st)
LGP=dict(objective="binary",metric="auc",verbose=-1,learning_rate=0.05,num_leaves=63,feature_fraction=0.8,bagging_fraction=0.8,bagging_freq=1,min_child_samples=50)
def lgb_oof_on(mask, seed):
    # mask=True 행에서만 5-fold 학습 -> 그 행들 OOF (나머지 NaN)
    idx=np.where(mask)[0]; Xm=Xb.iloc[idx]; ym=y[idx]; catf=[c for c in CATF if c in Xm.columns]
    o=np.full(N,np.nan)
    for tri,vai in StratifiedKFold(5,shuffle=True,random_state=seed).split(Xm,ym):
        m=lgb.train(dict(LGP,seed=seed),lgb.Dataset(Xm.iloc[tri],ym[tri],categorical_feature=catf),1500,
                    valid_sets=[lgb.Dataset(Xm.iloc[vai],ym[vai],categorical_feature=catf)],
                    callbacks=[lgb.early_stopping(80,verbose=False),lgb.log_evaluation(0)])
        o[idx[vai]]=m.predict(Xm.iloc[vai])
    return o
from sklearn.model_selection import KFold
# ── 재학습 캐시: seg_oof/STEP A/STEP B가 같은 (mask,seed)를 반복 호출 → seed당 1회만 학습(로컬 M1·GPU 없음)
_pos_cache={}; _full_cache={}
def pos_oof(seed):   # 이식>0 전용 5-fold OOF (해당 행만 반환)
    if seed not in _pos_cache: _pos_cache[seed]=lgb_oof_on(mpos,seed)[mpos]
    return _pos_cache[seed]
def full_oof(seed):  # 전체(비분리) 5-fold OOF
    if seed not in _full_cache: _full_cache[seed]=lgb_oof_on(np.ones(N,bool),seed)
    return _full_cache[seed]
print("tf_tree",Xb.shape,"| 세그먼트 학습기 준비")

tf_tree (256351, 72) | 세그먼트 학습기 준비


## 3. STEP A — 세그먼트 vs 같은-파이프라인 비분리 baseline (세그먼트 효과 격리)
seg = 이식>0 전용 lgb OOF + 이식=0 상수. full = 전체 lgb OOF(같은 파이프라인). seg−full 증분 CI로 세그먼트 효과만 격리. 멀티시드.

In [4]:
def seg_oof(seed):
    # 이식>0: 전용모델 OOF. 이식=0: train-only "폴드별" 상수(누수 회피 — 전역 y[m0].mean() 대신
    # 각 검증폴드는 나머지 폴드의 train 평균만 사용. full(정상 OOF)과 공정 비교되도록).
    o=np.full(N,np.nan); o[mpos]=pos_oof(seed)
    m0idx=np.where(m0)[0]; ym0=y[m0idx]
    o[m0idx]=ym0.mean()  # 단일클래스/소표본 폴드 fallback
    for tri,vai in KFold(5,shuffle=True,random_state=seed).split(m0idx):
        o[m0idx[vai]]=ym0[tri].mean()
    return o
def paired_ci(pa,pb,n=2000,seed=0):
    r=np.random.default_rng(seed); ds=np.empty(n)
    for i in range(n):
        ix=r.integers(0,N,N); ds[i]=roc_auc_score(y[ix],pa[ix])-roc_auc_score(y[ix],pb[ix])
    return float(ds.mean()),float(np.percentile(ds,2.5)),float(np.percentile(ds,97.5))
print("=== STEP A — 세그먼트 vs 비분리(같은 파이프라인) ===")
inc_ms=[]
for s in SEEDS:
    full=full_oof(s); seg=seg_oof(s)
    a_full=roc_auc_score(y,full); a_seg=roc_auc_score(y,seg); inc,lo,hi=paired_ci(seg,full); inc_ms.append(inc)
    print(f"seed{s}: full={a_full:.5f} | seg={a_seg:.5f} | seg−full={inc:+.5f} CI[{lo:+.5f},{hi:+.5f}] {'✅' if lo>0 else '—'}")
mu,sd=np.mean(inc_ms),np.std(inc_ms)
print(f"\n멀티시드 seg−full {mu:+.5f}±{sd:.5f} → {'✅ 세그먼트 효과 있음' if (mu>0 and abs(mu)>sd) else '— 세그먼트 무효(이미 흡수/희석 없음)'}")
print(f"참조: seg(42)={roc_auc_score(y,seg_oof(42)):.5f} vs 운영 {auc_all:.5f} (파이프라인 달라 절대비교 X)")

=== STEP A — 세그먼트 vs 비분리(같은 파이프라인) ===


seed42: full=0.73944 | seg=0.73720 | seg−full=-0.00222 CI[-0.00268,-0.00177] —


seed7: full=0.73928 | seg=0.73712 | seg−full=-0.00215 CI[-0.00259,-0.00172] —


seed2024: full=0.73962 | seg=0.73700 | seg−full=-0.00262 CI[-0.00307,-0.00219] —

멀티시드 seg−full -0.00233±0.00021 → — 세그먼트 무효(이미 흡수/희석 없음)
참조: seg(42)=0.73720 vs 운영 0.74056 (파이프라인 달라 절대비교 X)


## 4. STEP B — 천장 재측정(이식>0) + 세그먼트 스프레드

In [5]:
pos_only=pos_oof(42); auc_pos_seg=roc_auc_score(y[mpos],pos_only)
# 공정비교: 같은 단일 lgb 파이프라인의 이식>0 부분집합(=full을 이식>0로 자른 것). 블렌드 대비 모델수 교란 없음.
auc_pos_full_single=roc_auc_score(y[mpos], full_oof(42)[mpos])
print(f"이식>0 전용모델 AUC={auc_pos_seg:.5f} vs 비분리 단일모델 이식>0 부분집합 AUC={auc_pos_full_single:.5f} (차 {auc_pos_seg-auc_pos_full_single:+.5f}) ← 공정(단일 vs 단일)")
print(f"  (참조) 운영 블렌드 이식>0 부분집합 AUC={auc_pos:.5f} — 멤버수 달라 절대비교 X")
print("→", "전용모델↑: deterministic 행이 희석 → 세그먼트 학습 레버" if auc_pos_seg>auc_pos_full_single+0.0005 else "비슷: 천장은 데이터 본연(이 축 닫힘, 공정 네거티브)")
def spread(mask,label):
    mm=np.asarray(mask)&mpos
    if mm.sum()<50: return f"  {label}: n={int(mm.sum())} (소표본)"
    return f"  {label}: n={int(mm.sum())} ({100*mm.mean():.1f}%) 성공률={y[mm].mean():.4f} Δ={100*(y[mm].mean()-y[mpos].mean()):+.2f}%p"
def col_is(c,val): return (train[c].astype(str)==val).values if c in train else np.zeros(N,bool)
print("\n이식>0 내부 세그먼트 스프레드:")
print(spread((pd.to_numeric(train.get('신선 배아 사용 여부'),errors='coerce')==1).values,"신선배아"))
print(spread((pd.to_numeric(train.get('동결 배아 사용 여부'),errors='coerce')==1).values,"동결배아"))
print(spread(col_is("시술 유형","IVF"),"IVF")); print(spread(col_is("시술 유형","DI"),"DI"))
print(spread(col_is("난자 출처","본인 제공"),"본인난자")); print(spread(col_is("난자 출처","기증 제공"),"기증난자"))

이식>0 전용모델 AUC=0.67307 vs 비분리 단일모델 이식>0 부분집합 AUC=0.67311 (차 -0.00004) ← 공정(단일 vs 단일)
  (참조) 운영 블렌드 이식>0 부분집합 AUC=0.67456 — 멤버수 달라 절대비교 X
→ 비슷: 천장은 데이터 본연(이 축 닫힘, 공정 네거티브)

이식>0 내부 세그먼트 스프레드:
  신선배아: n=176438 (68.8%) 성공률=0.3187 Δ=+1.24%p
  동결배아: n=37281 (14.5%) 성공률=0.2469 Δ=-5.94%p
  IVF: n=213516 (83.3%) 성공률=0.3062 Δ=+0.00%p
  DI: n=0 (소표본)
  본인난자: n=199216 (77.7%) 성공률=0.3033 Δ=-0.30%p
  기증난자: n=14300 (5.6%) 성공률=0.3476 Δ=+4.13%p


## 5. STEP C — 합류 준비 (그리드 v3 결과 나온 뒤 ρ 상관)

In [6]:
seg_final=seg_oof(42); os.makedirs("./seg_out",exist_ok=True)
pd.DataFrame({"oof_seg":seg_final,"y":y}).to_csv("./seg_out/oof_segment.csv",index=False)
print("💾 ./seg_out/oof_segment.csv (그리드 v3 OOF와 ρ 상관으로 중복/보완 판정)")
print("\n=== 요약 ===")
print(f"STEP0: 전체 {auc_all:.5f} / 이식>0 {auc_pos:.5f} / 흡수(완화)={ABSORBED}")
print(f"STEP A: seg−full {np.mean(inc_ms):+.5f}±{np.std(inc_ms):.5f}")
print(f"STEP B: 이식>0 전용 {auc_pos_seg:.5f} vs 비분리 단일 {auc_pos_full_single:.5f} (차 {auc_pos_seg-auc_pos_full_single:+.5f}, 공정비교)")

💾 ./seg_out/oof_segment.csv (그리드 v3 OOF와 ρ 상관으로 중복/보완 판정)

=== 요약 ===
STEP0: 전체 0.74056 / 이식>0 0.67456 / 흡수(완화)=True
STEP A: seg−full -0.00233±0.00021
STEP B: 이식>0 전용 0.67307 vs 비분리 단일 0.67311 (차 -0.00004, 공정비교)
